<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Human-in-the-Loop
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M18-Human-in-the-Loop"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

***Checkpointing & Sessions*** führte Checkpointing ein – der State bleibt über Aufrufe hinweg erhalten.  
**Dieses Modul** ergänzt das entscheidende Kontrollmuster: **der Mensch wird in den Ablauf eingebunden.**

Ohne HITL läuft ein Agent vollautomatisch – inklusive kritischer Aktionen.  
Mit HITL pausiert der Graph an definierten Punkten und wartet auf menschliche Entscheidung.

**Typische HITL-Szenarien:**

| Szenario | Kritische Aktion | HITL-Schritt |
|----------|-----------------|---------------|
| **Content-Freigabe** | Text veröffentlichen | Mensch liest + genehmigt Entwurf |
| **Tool-Genehmigung** | E-Mail senden, Datei löschen | Mensch bestätigt Tool-Aufruf |
| **Daten-Review** | Datenbankschreibzugriff | Mensch prüft Vorschau |
| **Eskalation** | Unklare Anfragen | Mensch übernimmt manuell |

**Zwei Wege, HITL umzusetzen:**

| Methode | Beschreibung | Einsatz |
|---------|-------------|--------|
| **`interrupt()`** in Node | Node fragt aktiv nach | Approval, Edit, Eskalation |
| **`interrupt_before`** | Graph pausiert deklarativ vor Node | Tool-Genehmigung |

> **Voraussetzung:** Ein Checkpointer (z.B. `MemorySaver`) ist zwingend erforderlich –  
> ohne gespeicherten State kann der Graph nicht fortgesetzt werden.

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: HITL Workflow</font> </br></p>

diagram = '''
flowchart TD
    START([START]) --> A["🤖 Agent Node"]
    A --> INT["⏸️ interrupt()\nGraph pausiert"]
    INT -->|"Checkpoint gespeichert"| H["👤 Mensch\nprüft & entscheidet"]
    H -->|"Command(resume='ja')"| R["▶️ Graph\nfortsätzt"]
    H -->|"Command(resume='nein')"| N["❌ Abbruch"]
    R --> END([END])
    N --> END

    style INT fill:#FF9800,color:#fff
    style H   fill:#2196F3,color:#fff
    style R   fill:#4CAF50,color:#fff
    style N   fill:#F44336,color:#fff
'''

mermaid(diagram, width=750)

# 2 | HITL Konzept
---

**Wie funktioniert `interrupt()`?**

```python
from langgraph.types import interrupt, Command

def genehmigung_node(state):
    # Pausiert den Graphen und gibt den Wert an den Aufrufer
    antwort = interrupt({"frage": "Genehmigen?", "daten": state["entwurf"]})
    return {"genehmigt": antwort == "ja"}
```

**Ablauf Schritt für Schritt:**

1. `graph.invoke(input, config=cfg)` – startet Ausführung
2. Node ruft `interrupt(wert)` auf – Graph pausiert, Checkpoint wird gespeichert
3. `invoke()` gibt Ergebnis mit `__interrupt__`-Schlüssel zurück
4. Mensch sieht den Interrupt-Wert und entscheidet
5. `graph.invoke(Command(resume=entscheidung), config=cfg)` – Graph setzt fort
6. `interrupt()` gibt den Resume-Wert zurück – Node verarbeitet ihn

**Drei HITL-Patterns im Überblick:**

| Pattern | Ablauf | Wann |
|---------|--------|------|
| **Approval** | Agent schlägt vor → Mensch genehmigt/lehnt ab | Irreversible Aktionen |
| **Edit** | Agent erstellt Entwurf → Mensch ändert → Weiter | Content-Erstellung |
| **Tool-Freigabe** | Agent plant Tool → Mensch bestätigt → Ausführen | Externe API-Aufrufe |

> **Wichtig:** `interrupt()` kann mehrfach im selben Graphen aufgerufen werden –  
> jedes Mal wird ein neuer Checkpoint gespeichert und der Mensch befragt.

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Drei HITL-Patterns</font> </br></p>

diagram = '''
flowchart LR
    subgraph APP["🟢 Approval-Pattern"]
        direction TB
        A1["Agent\nVorschlag"] --> A2["interrupt()"]
        A2 -->|ja| A3["✅ Ausführen"]
        A2 -->|nein| A4["❌ Abbruch"]
    end

    subgraph EDIT["🟡 Edit-Pattern"]
        direction TB
        E1["Agent\nEntwurf"] --> E2["interrupt()"]
        E2 -->|"Korrektur"| E3["✅ Geändert\nWeiter"]
    end

    subgraph TOOL["🟠 Tool-Freigabe"]
        direction TB
        T1["Agent"] --> T2["interrupt()"]
        T2 -->|genehmigt| T3["🔧 ToolNode"]
        T2 -->|abgelehnt| T4["❌ Ende"]
        T3 --> T1
    end
'''

mermaid(diagram, width=980)

# 3 | interrupt() und Approval-Pattern
---

Das **Approval-Pattern** ist das häufigste HITL-Muster:  
Ein Agent erstellt einen Vorschlag – der Mensch genehmigt oder lehnt ab.

> **In diesem Notebook entscheiden Sie selbst!**  
> `input()` pausiert die Zelle so lange, bis Sie tippen und Enter drücken –  
> genau so wartet ein echter Workflow auf eine menschliche Entscheidung.

**Beispiel:** Redaktions-Workflow
- `entwurf_node` – LLM erstellt Marketing-Text
- `genehmigung_node` – `interrupt()` pausiert, zeigt Entwurf
- `veroeffentlichen_node` – veröffentlicht oder bricht ab

```python
def genehmigung_node(state):
    antwort = interrupt({
        "frage":   "Entwurf veröffentlichen?",
        "entwurf": state["entwurf"]
    })
    return {"genehmigt": str(antwort).lower() == "ja"}

# Aufruf-Seite: Graph pausiert, bis input() eine Antwort liefert
result = graph.invoke(input_data, config=cfg)          # → läuft bis interrupt()
entscheidung = input("Ihre Entscheidung (ja/nein): ")  # ← Zelle wartet hier
result = graph.invoke(Command(resume=entscheidung), config=cfg)
```

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from IPython.display import Image as IPImage

# State fuer den Redaktions-Workflow
class RedaktionsState(TypedDict):
    thema:     str   # Vorgabe des Nutzers
    entwurf:   str   # LLM-generierter Text
    genehmigt: bool  # Menschliche Entscheidung
    ergebnis:  str   # Finale Ausgabe

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.7)

def entwurf_node(state: RedaktionsState) -> dict:
    """LLM erstellt kompakten Marketing-Text."""
    prompt = [
        SystemMessage("Du erstellst prägnante Marketing-Texte auf Deutsch. Max. 3 Sätze."),
        HumanMessage(f"Erstelle einen Marketing-Text für: {state['thema']}"),
    ]
    response = llm.invoke(prompt)
    return {"entwurf": response.content}

def genehmigung_node(state: RedaktionsState) -> dict:
    """Pausiert den Graphen und wartet auf menschliche Genehmigung."""
    antwort = interrupt({
        "frage":   "Entwurf veröffentlichen? (ja/nein)",
        "entwurf": state["entwurf"],
    })
    return {"genehmigt": str(antwort).lower().strip() == "ja"}

def veroeffentlichen_node(state: RedaktionsState) -> dict:
    """Veröffentlicht oder bricht ab – je nach Genehmigung."""
    if state["genehmigt"]:
        return {"ergebnis": f"✅ Veröffentlicht: {state['entwurf']}"}
    return {"ergebnis": "❌ Entwurf abgelehnt – nicht veröffentlicht."}

# Graph aufbauen
builder_r = StateGraph(RedaktionsState)
builder_r.add_node("entwurf",       entwurf_node)
builder_r.add_node("genehmigung",   genehmigung_node)
builder_r.add_node("veroeffentlichen", veroeffentlichen_node)

builder_r.add_edge(START,          "entwurf")
builder_r.add_edge("entwurf",      "genehmigung")
builder_r.add_edge("genehmigung",  "veroeffentlichen")
builder_r.add_edge("veroeffentlichen", END)

memory_r       = MemorySaver()
redaktions_graph = builder_r.compile(checkpointer=memory_r)
print("✅ Redaktions-Graph kompiliert")

In [ ]:
try:
    display(IPImage(redaktions_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Graph-Visualisierung benoetigt playwright/pygraphviz)")

In [ ]:
import uuid

run_cfg = {"run_name": "Redaktion-Approval", "tags": ["m17", "approval"]}
# Eindeutige thread_id verhindert Konflikte beim Mehrfach-Ausführen
cfg1 = {"configurable": {"thread_id": f"redaktion-{uuid.uuid4().hex[:6]}"}, **run_cfg}

# ── Schritt 1: LLM erstellt Entwurf ──────────────────────────────────────────
result = redaktions_graph.invoke(
    {"thema": "unsere neue KI-Lernplattform",
     "entwurf": "", "genehmigt": False, "ergebnis": ""},
    config=cfg1
)

# ── Schritt 2: Ihre echte Entscheidung ───────────────────────────────────────
if "__interrupt__" in result:
    val = result["__interrupt__"][0].value
    mprint(f"## ⏸ Graph pausiert – Ihre Entscheidung ist gefragt\n\n"
           f"**{val['frage']}**\n\n"
           f"**Entwurf:**\n> {val['entwurf']}")

    # ★ Zelle wartet hier auf Ihre Eingabe
    entscheidung = input("\nIhre Entscheidung (ja / nein) → ").strip().lower()
    if entscheidung not in ("ja", "nein"):
        entscheidung = "nein"
        print("  (Unbekannte Eingabe – als 'nein' gewertet)")

    # ── Schritt 3: Graph mit Ihrer Entscheidung fortsetzen ──────────────────
    result = redaktions_graph.invoke(Command(resume=entscheidung), config=cfg1)
    mprint(f"\n**Ergebnis:** {result['ergebnis']}")

# 4 | HITL implementieren: Edit-Pattern
---

Das **Edit-Pattern** ermöglicht nicht nur Ja/Nein, sondern eine **inhaltliche Änderung**:

1. Agent erstellt Entwurf
2. Mensch sieht den Entwurf und schickt eine korrigierte Version
3. Graph arbeitet mit der korrigierten Version weiter

> **In diesem Notebook:** Sie tippen Ihren eigenen Text ein – oder drücken Enter,  
> um den Originalentwurf zu übernehmen.

**Option A – Resume mit geändertem Wert (empfohlen):**

```python
def edit_node(state):
    korrektur = interrupt({"entwurf": state["entwurf"]})
    neuer_entwurf = str(korrektur).strip() or state["entwurf"]  # leer → original
    return {"entwurf": neuer_entwurf, "genehmigt": True}

# Aufruf-Seite
result   = graph.invoke(input_data, config=cfg)
entwurf  = result["__interrupt__"][0].value["entwurf"]
ihr_text = input("Ihr Text (Enter = übernehmen): ").strip()
result   = graph.invoke(Command(resume=ihr_text or entwurf), config=cfg)
```

**Option B – `update_state()` vor dem Resume:**

```python
graph.update_state(cfg, {"entwurf": "Korrigierter Text..."}, as_node="edit")
graph.invoke(Command(resume="ok"), config=cfg)
```

**Empfehlung:** Option A ist klarer und vollständig in LangSmith tracebar.

In [ ]:
# === Edit-Pattern: Sie verbessern den Entwurf ===

def edit_node(state: RedaktionsState) -> dict:
    """Entwurf an den Menschen senden; Korrektur kommt via Command(resume=...)."""
    korrektur = interrupt({
        "nachricht": "Bitte den Entwurf prüfen und ggf. ändern:",
        "entwurf":   state["entwurf"],
    })
    neuer_entwurf = str(korrektur).strip() or state["entwurf"]
    return {"entwurf": neuer_entwurf, "genehmigt": True}

builder_e = StateGraph(RedaktionsState)
builder_e.add_node("entwurf",         entwurf_node)
builder_e.add_node("edit",             edit_node)
builder_e.add_node("veroeffentlichen", veroeffentlichen_node)
builder_e.add_edge(START,      "entwurf")
builder_e.add_edge("entwurf",  "edit")
builder_e.add_edge("edit",     "veroeffentlichen")
builder_e.add_edge("veroeffentlichen", END)

memory_e   = MemorySaver()
edit_graph = builder_e.compile(checkpointer=memory_e)

cfg_e = {"configurable": {"thread_id": f"edit-{uuid.uuid4().hex[:6]}"},
         "run_name": "Redaktion-Edit", "tags": ["m17", "edit"]}

# ── Schritt 1: LLM erstellt Entwurf ──────────────────────────────────────────
result = edit_graph.invoke(
    {"thema": "unser neuer Kundenservice-Chat",
     "entwurf": "", "genehmigt": False, "ergebnis": ""},
    config=cfg_e
)

# ── Schritt 2: Ihre echte Eingabe ────────────────────────────────────────────
if "__interrupt__" in result:
    val = result["__interrupt__"][0].value
    mprint(f"## ⏸ Entwurf zur Prüfung\n\n> {val['entwurf']}")

    print("\n★ Eigenen Text eintippen und Enter drücken")
    print("  (Nur Enter = Originalentwurf unverändert übernehmen)")
    eigener_text = input("\nIhr Text → ").strip()

    resume_text = eigener_text if eigener_text else val["entwurf"]
    if eigener_text:
        print("  → Ihr Text wird verwendet.")
    else:
        print("  → Originalentwurf übernommen.")

    # ── Schritt 3: Fortsetzen ────────────────────────────────────────────────
    result = edit_graph.invoke(Command(resume=resume_text), config=cfg_e)
    mprint(f"\n**Ergebnis:** {result['ergebnis']}")

# 5 | Tool-Call-Genehmigung
---

Viele Agenten können **externe Aktionen** ausführen – E-Mails senden, Dateien ändern,  
APIs aufrufen. HITL-Freigabe vor dem Tool-Aufruf verhindert unerwünschte Effekte.

**Architektur:**
- `agent_node` – LLM plant Tool-Aufrufe
- `freigabe_node` – `interrupt()` zeigt geplante Tools, wartet auf Genehmigung
- `tools_node` – führt genehmigte Tools aus
- `abbruch_node` – bei Ablehnung: saubere Beendigung

**Alternativ: `interrupt_before` (deklarativ)**

```python
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]   # Graph pausiert automatisch VOR 'tools'
)
# Resume: graph.invoke(None, config=cfg)  – kein Command nötig
```

> **Empfehlung:** `interrupt()` im Node für komplexe Prüflogik;  
> `interrupt_before` für einfache Pause-vor-Ausführung-Szenarien.

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Tool-Call-Genehmigung</font> </br></p>

diagram = '''
flowchart TD
    START([START]) --> AGENT["🤖 Agent Node\nllm.bind\_tools(...)"]

    AGENT -->|"tool\_calls vorhanden"| FREIG["⏸️ Freigabe-Node\ninterrupt()"]
    AGENT -->|"kein tool\_call"| END([END])

    FREIG -->|"genehmigt"| TOOLS["🔧 Tool Node"]
    FREIG -->|"abgelehnt"| ABBRUCH["❌ Abbruch-Node"]

    TOOLS -->|"Tool-Ergebnisse"| AGENT
    ABBRUCH --> END

    style FREIG   fill:#FF9800,color:#fff
    style TOOLS   fill:#2196F3,color:#fff
    style ABBRUCH fill:#F44336,color:#fff
'''

mermaid(diagram, width=700)

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

@tool
def sende_email(empfaenger: str, betreff: str, inhalt: str) -> str:
    """Sendet eine E-Mail an den Emfaenger."""
    return f"E-Mail an {empfaenger} gesendet: '{betreff}'"

@tool
def erstelle_bericht(titel: str, inhalt: str) -> str:
    """Erstellt einen Bericht mit Titel und Inhalt."""
    return f"Bericht '{titel}' erstellt ({len(inhalt)} Zeichen)"

@tool
def loesche_datei(pfad: str) -> str:
    """Loescht eine Datei am angegebenen Pfad."""
    return f"Datei '{pfad}' geloescht"

hitl_tools    = [sende_email, erstelle_bericht, loesche_datei]
llm_mit_tools = init_chat_model("openai:gpt-4o-mini", temperature=0.0).bind_tools(hitl_tools)

# State
class FreigabeState(TypedDict):
    messages:  Annotated[list, add_messages]
    genehmigt: bool

def agent_node(state: FreigabeState) -> dict:
    """Ruft LLM mit gebundenen Tools auf."""
    response = llm_mit_tools.invoke(state["messages"])
    return {"messages": [response]}

def freigabe_node(state: FreigabeState) -> dict:
    """Zeigt geplante Tool-Aufrufe und wartet auf Genehmigung."""
    letzter_msg = state["messages"][-1]
    tool_namen  = [tc["name"] for tc in letzter_msg.tool_calls]
    antwort = interrupt({
        "nachricht":  f"Geplante Tool-Aufrufe: {', '.join(tool_namen)}",
        "anweisung":  "Genehmigen? (ja/nein)",
        "tool_calls": letzter_msg.tool_calls,
    })
    return {"genehmigt": str(antwort).lower().strip() == "ja"}

def freigabe_router(state: FreigabeState) -> str:
    return "tools" if state.get("genehmigt") else "abbruch"

def agent_router(state: FreigabeState) -> str:
    """Weiterleitung: Tool-Calls vorhanden → freigabe; sonst → __end__."""
    letzter = state["messages"][-1] if state["messages"] else None
    if letzter and hasattr(letzter, "tool_calls") and letzter.tool_calls:
        return "freigabe"
    return "__end__"   # String statt END-Konstante → Visualisierung korrekt

def abbruch_node(state: FreigabeState) -> dict:
    return {"messages": [AIMessage(content="Tool-Ausführung wurde abgelehnt.")]}

# Graph – explizite path_maps für korrekte Visualisierung
builder_f = StateGraph(FreigabeState)
builder_f.add_node("agent",    agent_node)
builder_f.add_node("freigabe", freigabe_node)
builder_f.add_node("tools",    ToolNode(hitl_tools))
builder_f.add_node("abbruch",  abbruch_node)

builder_f.add_edge(START, "agent")

# path_map explizit angeben → alle Kanten für draw_mermaid_png sichtbar
builder_f.add_conditional_edges(
    "agent",
    agent_router,
    {"freigabe": "freigabe", "__end__": END}
)
builder_f.add_conditional_edges(
    "freigabe",
    freigabe_router,
    {"tools": "tools", "abbruch": "abbruch"}
)
builder_f.add_edge("tools",   "agent")
builder_f.add_edge("abbruch", END)

memory_f       = MemorySaver()
freigabe_graph = builder_f.compile(checkpointer=memory_f)
print("✅ Tool-Freigabe-Graph kompiliert")

In [ ]:
try:
    display(IPImage(freigabe_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Graph-Visualisierung benoetigt playwright/pygraphviz)")

In [ ]:
run_cfg_f = {"run_name": "Tool-Freigabe", "tags": ["m17", "tool-approval"]}
cfg_f = {"configurable": {"thread_id": f"freigabe-{uuid.uuid4().hex[:6]}"}, **run_cfg_f}

# ── Schritt 1: Anfrage stellen – LLM plant Tool-Aufruf ───────────────────────
result = freigabe_graph.invoke(
    {"messages": [HumanMessage("Sende eine Zusammenfassung an chef@firma.de")],
     "genehmigt": False},
    config=cfg_f
)

# ── Schritt 2: Geplante Tools anzeigen + Ihre Entscheidung ───────────────────
if "__interrupt__" in result:
    val = result["__interrupt__"][0].value

    # Geplante Aufrufe detailliert ausgeben
    tool_details = "\n".join(
        f"  • **{tc['name']}**"
        + (f"({', '.join(f'`{k}`={v!r}' for k, v in tc.get('args', {}).items())})"
           if tc.get("args") else "()")
        for tc in val.get("tool_calls", [])
    )
    mprint(f"## ⏸ Tool-Freigabe erforderlich\n\n"
           f"**{val['nachricht']}**\n\n"
           f"Geplante Aufrufe:\n{tool_details}\n\n"
           f"*{val['anweisung']}*")

    # ★ Zelle wartet auf Ihre Eingabe
    entscheidung = input("\nIhre Entscheidung (ja / nein) → ").strip().lower()
    if entscheidung not in ("ja", "nein"):
        entscheidung = "nein"
        print("  (Unbekannte Eingabe – als 'nein' gewertet)")

    # ── Schritt 3: Graph fortsetzen ──────────────────────────────────────────
    result = freigabe_graph.invoke(Command(resume=entscheidung), config=cfg_f)
    mprint(f"\n**Ergebnis:** {result['messages'][-1].content}")

# 6 | LangSmith: HITL-Traces
---

HITL-Workflows erzeugen in LangSmith besondere Trace-Strukturen:  
Ein unterbrochener Graph erscheint als **zweistufiger Run** –  
invoke_1 (bis interrupt) + invoke_2 (nach resume).

**Was in LangSmith sichtbar ist:**

| Element | Beschreibung |
|---------|-------------|
| **Interrupted Run** | Run-Status `interrupted` bis zum Resume |
| **Resume Run** | Neuer Run mit gleichem `thread_id`, angedockt an den ersten |
| **Interrupt-Wert** | In den Metadaten des Node-Spans sichtbar |
| **Human Decision** | Resume-Wert erscheint als Input des zweiten Runs |

**Best Practices für HITL-Tracing:**

```python
cfg = {
    "configurable": {"thread_id": "hitl-session-42"},
    "run_name":  "HITL-Content-Freigabe",
    "tags":      ["m17", "hitl", "approval", "marketing"],
    "metadata":  {
        "user_id":    "user-42",
        "workflow":   "content-approval",
        "version":    "1.0",
    }
}
```

> **Tipp:** Verwende aussagekräftige `run_name`-Werte für HITL-Runs,  
> z.B. `"Freigabe-Entwurf"` statt `"invoke"`. Das erleichtert das Debugging  
> erheblich, wenn viele parallele HITL-Sessions aktiv sind.  
> *LangSmith Evaluations Basics* zeigt die vollständige Trace-Analyse.

In [ ]:
# LangSmith-optimiertes HITL-Beispiel
# ──────────────────────────────────────────────────────────────────────────────
# Beobachten Sie in LangSmith:
#   Run 1 → Status "interrupted" (bis zur Eingabe)
#   Run 2 → Status "success"    (nach Command(resume=...))
# ──────────────────────────────────────────────────────────────────────────────
cfg_ls = {
    "configurable": {"thread_id": f"hitl-ls-{uuid.uuid4().hex[:6]}"},
    "run_name": "HITL-Marketing-Freigabe",
    "tags":     ["m17", "hitl", "langsmith-demo"],
    "metadata": {
        "user_id":  "trainer-42",
        "workflow": "content-approval",
    }
}

result = redaktions_graph.invoke(
    {"thema": "LangSmith-Tracing", "entwurf": "", "genehmigt": False, "ergebnis": ""},
    config=cfg_ls
)

if "__interrupt__" in result:
    val = result["__interrupt__"][0].value
    mprint(f"## ⏸ LangSmith sieht diesen Run als 'interrupted'\n\n"
           f"**Entwurf:**\n> {val['entwurf']}")

    # ★ Echte Eingabe – LangSmith protokolliert Ihre Entscheidung als Resume-Input
    entscheidung = input("\nIhre Entscheidung (ja / nein) → ").strip().lower()
    if entscheidung not in ("ja", "nein"):
        entscheidung = "nein"

result = redaktions_graph.invoke(Command(resume=entscheidung), config=cfg_ls)
print(f"\nErgebnis: {result['ergebnis'][:100]}")
print("\nIn LangSmith sichtbar:")
print(f"  Run 1  → 'HITL-Marketing-Freigabe'  Status: interrupted")
print(f"  Run 2  → Resume-Run                 Status: success")
print(f"  Input  → Command(resume='{entscheidung}')")
print(f"  Tags   → ['m17', 'hitl', 'langsmith-demo']")

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M18-Human-in-the-Loop", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Code-Review-Assistent mit HITL
</font></p>

Bauen Sie einen Agenten, der Code analysiert, Verbesserungsvorschläge erstellt und diese zur menschlichen Prüfung vorlegt, bevor sie angewendet werden.

**State:**
```python
class CodeReviewState(TypedDict):
    code:         str    # Eingabe-Code
    analyse:      str    # LLM-Analyse: Probleme gefunden
    vorschlaege:  str    # LLM-Verbesserungsvorschläge
    genehmigt:    bool   # Menschliche Freigabe
    finaler_code: str    # Verbesserter Code (nach Freigabe)
```

**Nodes und Routing:**
1. `analyse_node` – LLM analysiert den Code auf Probleme (Style, Security, Effizienz)
2. `vorschlag_node` – LLM erstellt konkreten Verbesserungsvorschlag
3. `review_node` – `interrupt()` legt Analyse + Vorschlag dem Menschen vor
4. `anwenden_node` – wendet Vorschlag an (bei Genehmigung) oder beendet (bei Ablehnung)

**Bonus:**
- Edit-Pattern: Mensch kann den Verbesserungsvorschlag selbst anpassen
- SqliteSaver kombinieren: Review-Sessions bleiben persistent
- Mehrere Iterationsrunden: Nach Ablehnung macht der Agent einen neuen Vorschlag (Qualitäts-Gate-Muster kombinieren)


---
### 🔍 Selbstcheck mit `assert`

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Implementieren Sie den Code-Review-Graphen mit `CodeReviewState` und `interrupt()` in den Zellen über diesem Block
2. Speichern Sie den kompilierten Graph in **`mein_graph`** (`mein_graph = builder.compile(checkpointer=...)`)
3. Führen Sie die Zelle unten aus — `interrupt()` und `Command(resume=...)` werden beide geprüft


In [ ]:
#@title ✅ Selbstcheck – HITL Code-Review  { display-mode: "form" }
# ► Speichere deinen Graph (mit Checkpointer + interrupt) in 'mein_graph'.

from langgraph.types import Command

_g   = mein_graph  # ← Variablennamen anpassen
_cfg = {"configurable": {"thread_id": "self-check-hitl"}}

assert hasattr(_g, "invoke"), \
    "❌ Graph hat kein invoke() – wurde builder.compile(checkpointer=...) aufgerufen?"

# Erster Aufruf: Graph sollte bei interrupt() pausieren
_g.invoke({
    "code":         "def add(a,b): return a+b",
    "analyse":      "",
    "vorschlaege":  "",
    "genehmigt":    False,
    "finaler_code": "",
}, config=_cfg)

_state = _g.get_state(_cfg)
assert len(_state.next) > 0, \
    "❌ Graph hat nicht gestoppt – interrupt() nicht aufgerufen?\n" \
    "   Prüfe: interrupt('...') im review_node"

print(f"✅ interrupt() hat ausgelöst – Graph wartet auf menschliche Freigabe")
print(f"   Nächster Node: {_state.next}")

# Resume mit Freigabe
_g.invoke(Command(resume={"genehmigt": True}), config=_cfg)
_state_final = _g.get_state(_cfg)
assert len(_state_final.next) == 0, \
    "❌ Graph nicht abgeschlossen nach Resume – prüfe anwenden_node"

print("✅ Resume erfolgreich – Graph vollständig durchgelaufen")
